# 01 · Prepare Training Data

This notebook prepares the data the R model is trained on, starting from raw bioactivity records in ChEMBL, and explores it:

1. Fetch raw IC50 records for human aromatase (CYP19A1, `CHEMBL1978`) from the ChEMBL REST API into `CHEMBL1978_RAW_ACTIVITIES`
2. Clean and deduplicate them into `CHEMBL1978_COMPOUNDS`, labelling compounds Active at pIC50 >= 6.5
3. Explore the pIC50 distribution, class balance, and Lipinski Rule of Five properties
4. Fetch the assay descriptions and source publications behind each measurement into `CHEMBL1978_ASSAYS` and `CHEMBL1978_DOCUMENTS`, for the agent's search index
5. Write `chembl1978_cleaned.csv` (the R training input), `dashboard_eda.csv`, and `dashboard_ro5.csv` to the app's stage

Run this before training the model. `02_score_compounds.ipynb` adds model predictions once the R service is running.

**Requirements:** run on a Container Runtime with an external access integration that allows `www.ebi.ac.uk`, `pypi.org`, and `files.pythonhosted.org` (`QSAR_NB_EAI` in `sql/setup.sql`).

## Configuration

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

DATABASE = "QSAR_DB"
SCHEMA = "PUBLIC"
STAGE = f"@{DATABASE}.{SCHEMA}.QSAR_STAGE"

TARGET_CHEMBL_ID = "CHEMBL1978"   # human aromatase (CYP19A1)
ACTIVE_THRESHOLD = 6.5            # pIC50 >= 6.5 (IC50 <= ~316 nM) is Active

session.use_database(DATABASE)
session.use_schema(SCHEMA)
print(session.get_current_database(), session.get_current_schema())

## Install RDKit

RDKit computes the Rule of Five descriptors. It installs from PyPI through the notebook's external access integration. The version is pinned because hydrogen-bond acceptor counts changed in later RDKit releases.

In [ ]:
%pip install -q rdkit==2025.09.3

## Fetch Raw Bioactivity Data

Query the ChEMBL REST API for binding-assay IC50 values reported with an exact (`=`) relation. Records are stored as-is so the cleaning step below can be rerun or audited.

In [ ]:
import time

import requests
import pandas as pd

CHEMBL_API = "https://www.ebi.ac.uk/chembl/api/data"


def get_json(url, params=None, retries=5):
    """GET a ChEMBL API page, retrying transient errors and non-JSON responses."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, params=params, timeout=120)
            resp.raise_for_status()
            return resp.json()
        except (requests.RequestException, ValueError) as err:
            if attempt == retries - 1:
                raise RuntimeError(f"ChEMBL API request failed after {retries} attempts: {url}") from err
            time.sleep(10 * (attempt + 1))

FIELDS = ["activity_id", "molecule_chembl_id", "canonical_smiles",
          "standard_value", "standard_units", "standard_relation", "pchembl_value",
          "assay_chembl_id", "document_chembl_id"]

release = get_json(f"{CHEMBL_API}/status.json")
print("ChEMBL release:", release["chembl_db_version"], release["chembl_release_date"])

params = {
    "target_chembl_id": TARGET_CHEMBL_ID,
    "standard_type": "IC50",
    "standard_relation": "=",
    "assay_type": "B",
    "only": ",".join(FIELDS),
    "limit": 1000,
}
records, url = [], f"{CHEMBL_API}/activity.json"
while url:
    page = get_json(url, params)
    records.extend(page["activities"])
    nxt = page["page_meta"]["next"]
    url, params = (f"https://www.ebi.ac.uk{nxt}" if nxt else None), None

raw_df = pd.DataFrame(records)[FIELDS].astype("string")
raw_df.columns = [c.upper() for c in raw_df.columns]
print(f"Raw records: {len(raw_df)}")

session.write_pandas(raw_df, "CHEMBL1978_RAW_ACTIVITIES",
                     auto_create_table=True, overwrite=True, table_type="")

## Clean and Deduplicate

Keep nM records with a SMILES string and a pChEMBL value, then keep the highest pIC50 per molecule. This is the curated, non-redundant set the R model is trained on. The result is saved as the `CHEMBL1978_COMPOUNDS` table.

In [ ]:
df = session.table("CHEMBL1978_RAW_ACTIVITIES").to_pandas()   # bounded: a few thousand raw records
df.columns = [c.lower() for c in df.columns]
df = df.sort_values("activity_id", key=lambda s: pd.to_numeric(s))   # ChEMBL API order

df = df[df["standard_units"] == "nM"]
df = df.dropna(subset=["canonical_smiles", "pchembl_value"])
df = df[(df["canonical_smiles"].str.len() > 0) & (df["canonical_smiles"] != "None")].copy()
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df["pchembl_value"] = pd.to_numeric(df["pchembl_value"], errors="coerce")
df = df.dropna(subset=["pchembl_value"])

# Keep the highest pIC50 measurement for each molecule; a stable sort keeps the
# earliest ChEMBL record when measurements tie, so reruns give the same result
compounds = (
    df.sort_values("pchembl_value", ascending=False, kind="stable")
      .drop_duplicates(subset="molecule_chembl_id", keep="first")
      .reset_index(drop=True)
)
compounds["activity_class"] = (compounds["pchembl_value"] >= ACTIVE_THRESHOLD).astype(int)
compounds["activity_label"] = compounds["activity_class"].map({1: "Active", 0: "Inactive"})

table_df = compounds[["molecule_chembl_id", "canonical_smiles", "standard_value",
                      "pchembl_value", "activity_class", "assay_chembl_id", "document_chembl_id"]].copy()
table_df.columns = [c.upper() for c in table_df.columns]
session.write_pandas(table_df, "CHEMBL1978_COMPOUNDS",
                     auto_create_table=True, overwrite=True, table_type="")
print(f"Compounds: {len(compounds)}")

In [ ]:
%%sql -r compound_summary
SELECT
    COUNT(*)                           AS N_COMPOUNDS,
    SUM(ACTIVITY_CLASS)                AS N_ACTIVE,
    COUNT(*) - SUM(ACTIVITY_CLASS)     AS N_INACTIVE,
    ROUND(MIN(PCHEMBL_VALUE), 2)       AS MIN_PIC50,
    ROUND(MAX(PCHEMBL_VALUE), 2)       AS MAX_PIC50
FROM CHEMBL1978_COMPOUNDS

The 2,751-compound training set has 1,061 actives. A newer ChEMBL release can change these counts.

About 130 molecules have more than one measurement at the same top pIC50. This notebook keeps the earliest record, so `standard_value` can differ slightly from the repository's `data/chembl1978_cleaned.csv` for those molecules. SMILES, pIC50, and activity class are identical, and the model and dashboard use only those columns.

## Fetch Assay and Publication Context

Each measurement comes from an assay with a free-text description (enzyme source, substrate, incubation time, detection method) and a source publication. Storing them lets the agent explain where a value came from and whether two measurements are comparable. ChEMBL data is licensed under CC BY-SA 3.0.

In [ ]:
def fetch_by_ids(endpoint, key, ids, fields, batch=50):
    rows = []
    ids = sorted(set(i for i in ids if i))
    for start in range(0, len(ids), batch):
        params = {f"{key}__in": ",".join(ids[start:start + batch]), "only": ",".join(fields), "limit": batch}
        rows.extend(get_json(f"{CHEMBL_API}/{endpoint}.json", params)[f"{endpoint}s"])
    return pd.DataFrame(rows, columns=fields)

ASSAY_FIELDS = ["assay_chembl_id", "description", "assay_type", "assay_cell_type",
                "assay_organism", "confidence_score", "document_chembl_id"]
DOC_FIELDS = ["document_chembl_id", "title", "journal", "year", "doi", "pubmed_id", "abstract"]

raw_ids = session.table("CHEMBL1978_RAW_ACTIVITIES").select("ASSAY_CHEMBL_ID", "DOCUMENT_CHEMBL_ID").distinct().to_pandas()
assays = fetch_by_ids("assay", "assay_chembl_id", raw_ids["ASSAY_CHEMBL_ID"].dropna(), ASSAY_FIELDS)
documents = fetch_by_ids("document", "document_chembl_id", raw_ids["DOCUMENT_CHEMBL_ID"].dropna(), DOC_FIELDS)

for name, frame in [("CHEMBL1978_ASSAYS", assays), ("CHEMBL1978_DOCUMENTS", documents)]:
    out = frame.astype("string")
    out.columns = [col.upper() for col in out.columns]
    session.write_pandas(out, name, auto_create_table=True, overwrite=True, table_type="")
    print(f"{name}: {len(out)} rows")


## Explore Bioactivity

The pIC50 distribution shows how activity is spread across the dataset, and the dashed line marks the Active threshold used for classification.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [3, 1]})
for label, color in [("Inactive", "#00bcd4"), ("Active", "#e040fb")]:
    ax1.hist(compounds.loc[compounds["activity_label"] == label, "pchembl_value"],
             bins=40, alpha=0.7, color=color, label=label)
ax1.axvline(ACTIVE_THRESHOLD, color="black", linestyle="--", linewidth=1)
ax1.set_xlabel("pIC50"); ax1.set_ylabel("Compounds"); ax1.set_title("pIC50 distribution"); ax1.legend()

counts = compounds["activity_label"].value_counts().reindex(["Active", "Inactive"])
ax2.bar(counts.index, counts.values, color=["#e040fb", "#00bcd4"])
for i, v in enumerate(counts.values):
    ax2.text(i, v, f"{v:,}", ha="center", va="bottom")
ax2.set_title("Class balance")
plt.tight_layout()
plt.show()

compounds["pchembl_value"].describe().round(2)

## Rule of Five Properties

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import Crippen, Descriptors, rdMolDescriptors

RDLogger.DisableLog("rdApp.*")

ro5_rows = []
for row in compounds.itertuples():
    mol = Chem.MolFromSmiles(row.canonical_smiles)
    if mol is None:
        continue
    ro5_rows.append({
        "pchembl_value":  row.pchembl_value,
        "mw":             round(Descriptors.MolWt(mol), 2),
        "logp":           round(Crippen.MolLogP(mol), 3),
        "hbd":            rdMolDescriptors.CalcNumHBD(mol),
        "hba":            rdMolDescriptors.CalcNumHBA(mol),
        "tpsa":           round(Descriptors.TPSA(mol), 2),
        "activity_label": row.activity_label,
    })
ro5_df = pd.DataFrame(ro5_rows)
print(f"Rule of Five rows: {len(ro5_df)} ({len(compounds) - len(ro5_df)} unparsable)")

Lipinski's Rule of Five flags compounds likely to have poor oral absorption: molecular weight above 500 Da, LogP above 5, more than 5 hydrogen-bond donors, or more than 10 acceptors. Comparing Active and Inactive compounds shows whether potency comes at the cost of drug-likeness.

In [ ]:
rules = {"MW <= 500": ro5_df["mw"] <= 500, "LogP <= 5": ro5_df["logp"] <= 5,
         "HBD <= 5": ro5_df["hbd"] <= 5, "HBA <= 10": ro5_df["hba"] <= 10}
compliance = (
    pd.DataFrame({name: ok for name, ok in rules.items()})
    .assign(activity_label=ro5_df["activity_label"])
    .groupby("activity_label").mean().mul(100).round(1).T
)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (prop, limit) in zip(axes, [("mw", 500), ("logp", 5), ("hbd", 5), ("hba", 10)]):
    data = [ro5_df.loc[ro5_df["activity_label"] == lbl, prop] for lbl in ["Active", "Inactive"]]
    ax.boxplot(data)
    ax.set_xticks([1, 2], ["Active", "Inactive"])
    ax.axhline(limit, color="red", linestyle="--", linewidth=1)
    ax.set_title(prop.upper())
plt.tight_layout()
plt.show()

compliance

## Write Training and Dashboard Files to the Stage

In [ ]:
import os
import tempfile

outputs = {
    "chembl1978_cleaned.csv": compounds[["molecule_chembl_id", "canonical_smiles", "standard_value",
                                         "pchembl_value", "activity_class"]],
    "dashboard_eda.csv":      compounds[["pchembl_value", "activity_label"]],
    "dashboard_ro5.csv":      ro5_df,
}

out_dir = tempfile.mkdtemp()
for name, frame in outputs.items():
    path = os.path.join(out_dir, name)
    frame.to_csv(path, index=False)
    session.file.put(path, STAGE, auto_compress=False, overwrite=True)
    print(f"{name:24s} {len(frame):5d} rows")

## Next Steps

Download `chembl1978_cleaned.csv` from the stage into the repository's `data/` folder and train the R model, as described in the guide. Once the R service is running, run `02_score_compounds.ipynb` to add model predictions for the dashboard.